# IA local grátis para o Manga Studio (Google Colab ou Kaggle)

Roda uma IA de tradução numa GPU grátis, **sem cota diária e sem bloqueio de conteúdo +18**.
O seu PC (MangaStudio.exe) continua fazendo OCR, limpeza e diagramação; só o texto vem para cá.

### No Google Colab
1. Menu **Ambiente de execução > Alterar o tipo de ambiente de execução > T4 GPU**.
2. Escolha o modelo na célula abaixo e clique em **Play**.
3. Em 3 a 6 minutos aparece uma URL `https://....trycloudflare.com`.

### No Kaggle (cerca de 30 horas de GPU por semana)
1. Em kaggle.com: **Create > New Notebook > File > Import Notebook** e cole o link deste arquivo no GitHub.
2. Painel direito: **Accelerator: GPU T4 x2** e **Internet: On** (precisa do celular verificado na conta).
3. Rode a célula abaixo.

### No Manga Studio
**Configurações > IA local (Ollama) > Endereço**: cole a URL, clique em **Testar conexão e velocidade** e salve.
Em **Traduzir**, deixe o motor em **Automático**: a IA local é usada primeiro e, se cair, o sistema segue pelo Gemini/Groq.

| Modelo | Quando usar |
|---|---|
| `qwen3:14b` | Melhor tradução que cabe na GPU grátis (padrão). |
| `qwen3:8b` | Mais rápido, um pouco menos caprichado. |
| `gemma3:12b` | Alternativa para comparar nas suas páginas. |

A sessão grátis cai depois de algumas horas ou se a aba ficar parada. A URL muda a cada vez que você roda a célula.


In [ ]:
# @title ▶ Ligar a IA local (clique em Play e espere a URL aparecer)
MODELO = "qwen3:14b"  # @param ["qwen3:14b", "qwen3:8b", "gemma3:12b"]
CONTEXTO = 16384  # @param [8192, 16384, 32768] {type:"raw"}
# qwen3:14b = melhor tradução que cabe na GPU grátis (T4). qwen3:8b = mais rápido. gemma3:12b = alternativa.

import json, os, re, shutil, subprocess, sys, time, urllib.request

EM_KAGGLE = os.path.exists("/kaggle")
EM_COLAB = "google.colab" in sys.modules
print(f"[*] Ambiente: {'Kaggle' if EM_KAGGLE else 'Google Colab' if EM_COLAB else 'Linux'}")

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("[!] Sem GPU. Colab: Ambiente de execução > Alterar tipo > T4 GPU. "
                     "Kaggle: Settings > Accelerator > GPU T4 x2 (e Internet: On).")
print(f"[+] GPU: {gpu}")

print("[*] [1/4] Instalando o Ollama...")
subprocess.run("apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null", shell=True)
if not shutil.which("ollama"):
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

print("[*] [2/4] Ligando o servidor...")
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH=str(CONTEXTO), OLLAMA_KEEP_ALIVE="60m",
           OLLAMA_NUM_PARALLEL="1", OLLAMA_FLASH_ATTENTION="1")
subprocess.run("pkill -f 'ollama serve'", shell=True)
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(1)

print(f"[*] [3/4] Baixando {MODELO} (2 a 5 minutos na primeira vez)...")
subprocess.run(["ollama", "pull", MODELO], env=env, check=True)

def _chat(texto, n=64):
    body = {"model": MODELO, "stream": False, "think": False, "keep_alive": "60m",
            "messages": [{"role": "user", "content": texto}],
            "options": {"temperature": 0.2, "num_predict": n, "num_ctx": CONTEXTO}}
    req = urllib.request.Request("http://127.0.0.1:11434/api/chat", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=600) as r:
        return json.loads(r.read().decode())

_chat("oi", 1)  # carrega o modelo na GPU
t0 = time.time()
res = _chat("Traduza para português do Brasil, só a tradução: \"I told you not to come here alone!\"")
tok_s = res.get("eval_count", 0) / max(1e-9, res.get("eval_duration", 1) / 1e9)
print(f"[+] Teste: {res['message']['content'].strip()}  ({tok_s:.0f} tokens/s)")

print("[*] [4/4] Abrindo o túnel público...")
if not shutil.which("cloudflared"):
    subprocess.run("wget -q -O /tmp/cf.deb https://github.com/cloudflare/cloudflared/releases/latest/download/"
                   "cloudflared-linux-amd64.deb && dpkg -i /tmp/cf.deb > /dev/null", shell=True, check=True)
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break

print("\n" + "=" * 70)
print("  IA LOCAL LIGADA. Cole esta URL no Manga Studio:")
print("  Configurações > IA local (Ollama) > Endereço")
print(f"\n      {url}\n")
print(f"  Modelo: {MODELO}  |  Contexto: {CONTEXTO} tokens  |  Velocidade: {tok_s:.0f} tokens/s")
print("  No Traduzir, deixe o motor em 'Automático' (ou escolha 'IA local').")
print("  Mantenha esta aba aberta. A URL muda toda vez que você rodar de novo.")
print("=" * 70)

while True:  # mantém a célula viva (e mostra que está tudo bem)
    time.sleep(300)
    vivo = server.poll() is None and tunnel.poll() is None
    print(f"[{time.strftime('%H:%M')}] {'IA local no ar' if vivo else 'algo caiu: rode a célula de novo'}: {url}")
